## Part A: Regular expressions (4 points)

In [3]:
import re
import nltk
nltk.download('gutenberg')
nltk.download('punkt_tab')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /Users/sakis/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/sakis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Complete the following function using the search function of the re library to find and return the year that a file in the gutenberg collection was written. Return the year as string or the empty string if not found. (2 points)

In [4]:
def get_year(file):
    contents = gutenberg.raw(file)
    re_year = r"\d{4}"
    rex = re.compile(re_year)
    res = rex.search(contents)
    
    if not res:
        return ""
    return res[0]

In [5]:
"""Testing that the function returns the correct results"""
assert get_year('carroll-alice.txt') == '1865'
assert get_year('shakespeare-hamlet.txt') == '1599'
assert get_year('milton-paradise.txt') == '1667'
assert get_year('melville-moby_dick.txt') == '1851'

Complete the following function using the findall function of the re library to compute and report the number of times some text appears in a file of the gutenberg collection. (1 point)

In [6]:
def find_mentions(text, file):
    contents = gutenberg.raw(file)
    needle = rf"{text}"
    rex = re.compile(needle)
    res = rex.findall(contents)
    return len(res)

In [7]:
"""Testing that the function returns the correct results"""
assert find_mentions('CHAPTER ', 'carroll-alice.txt') == 12
assert find_mentions('UNK', 'shakespeare-caesar.txt') == 0
assert find_mentions('Caesar', 'shakespeare-caesar.txt') == 227
assert find_mentions('girl', 'austen-emma.txt') == 60

Complete the following function to return the mean number of words in the sentences of a gutenberg file that is given as input. The number should be rounded to 2 decimals. (1 point)

In [8]:
def average_sentence_length(file):
    sentences = gutenberg.sents(file)
    total_tokens = sum([len(sentence) for sentence in sentences])
    return round(total_tokens/len(sentences), 2)

In [9]:
"""Testing that the function returns the correct results"""
assert average_sentence_length('shakespeare-caesar.txt') == 11.94
assert average_sentence_length('austen-emma.txt') == 24.82
assert average_sentence_length('milton-paradise.txt') == 52.31
assert average_sentence_length('melville-moby_dick.txt') == 25.93

AssertionError: 

## Part B: Byte-Pair Encoding (6 points)

Implement a function that returns the vocabulary learned by the byte-pair encoding algorithm given a corpus, after a number of iterations. The characters in the corpus, sorted in alphabetical order, will be the first members of the vocabulary, to be extended with as many tokens as the number of iterations given.

In [1]:
from nltk import word_tokenize

class Vocabulary:
    def __init__(self):
        self._items = set()

    @property
    def items(self):
        return sorted(list(self._items))

    @property
    def iset(self):
        return self._items
    
    def add(self, item):
        self._items.add(item)

    def add_many(self, items: list):
        self._items.update(set(items))

    def contains(self, item) -> bool:
        return item in self._items

def rewrite_word(word: list, pair: str) -> list:
    ret = []
    i = 0
    while i < len(word):
        if i < len(word) - 1 and f"{word[i]}{word[i+1]}" == pair:
            ret.append(pair)
            # we move one item over at the end of the loop, but
            # we need to move yet one more item here, 
            # to avoid double looking at word[i+1] twice
            i += 1
        else:
            ret.append(word[i])
        i += 1
    return ret

assert rewrite_word(["n", "e", "w", "e", "r"], "ew") == ["n", "ew", "e", "r"]
assert rewrite_word(["n", "ew", "e", "r"], "ewe") == ["n", "ewe", "r"]


def get_pair_counts(vocabed_words: list) -> dict:
    pair_counts = {}
    for vw in vocabed_words:
        # here we use zip to pair each word 'element' with the one next to it
        for pair in zip(vw, vw[1:]):
            k = f"{pair[0]}{pair[1]}"
            if k not in pair_counts:
                pair_counts[k] = 0
            pair_counts[k] += 1
    return pair_counts

assert get_pair_counts([["t", "a", "t", "a"], ["b", "a", "t"]]) == {"ta": 2, "at": 2, "ba": 1}
assert get_pair_counts([["t", "es", "t"], ["be", "a", "m"]]) == {"tes": 1, "est": 1, "bea": 1, "am": 1}

def bpe_train(text, iterations):
    # prefix all spaces with underscore, to help with the vocabulary
    text = text.replace(" ", " _")
    words = word_tokenize(text)
    # just to make sure word_tokenize didn't split the underscores from the words
    assert len(words) == len(text.split())
    
    vocab = Vocabulary()
    split_words = []
    for w in words:
        chars = list(w)
        split_words.append(chars)
        vocab.add_many(chars)

    
    for i in range(iterations):
        pair_counts = get_pair_counts(split_words)
        # use the pair with the highest count
        pair = max(pair_counts, key=lambda x: pair_counts[x])
        vocab.add(pair)
        split_words = [rewrite_word(w, pair) for w in split_words]
            
    return vocab.items

In [2]:
"""Testing that the function returns the correct results"""
corpus = "set new new renew reset renew"
assert set(bpe_train(corpus, 4)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re'])
assert set(bpe_train(corpus, 8)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re', 'se', 'set', '_new', '_renew'])
assert set(bpe_train(corpus, 9)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re', 'se', 'set', '_new', '_renew', '_reset']);

corpus = "hello world my large language model is coming after you beware"
assert set(bpe_train(corpus, 6)) == set(['_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u', 'w', 'y', 'el', '_m', '_l', '_la', 'ge', 'ng'])

corpus = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new"
assert set(bpe_train(corpus, 4)) == set(['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', 'er', '_n', '_ne', '_new'])